# RetailRocket Dataset — Sample Analysis

**Goal:** understand the three CSV files before building the pipeline.
**Scope:** 1,000-row samples from `02-data/samples/` (the full data lives in Azure Blob).
**Key questions:** What is an event? What's in each file? What traps will the ETL hit?

## 1. Setup — load the samples

The samples mirror the full files' schemas exactly (same headers, same types).

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (9, 4)

events = pd.read_csv("/projects/azure-dw-retailrocket/02-data/samples/events.csv")
cats = pd.read_csv("/projects/azure-dw-retailrocket/02-data/samples/category_tree.csv")
props = pd.read_csv("/projects/azure-dw-retailrocket/02-data/samples/item_properties.csv")

for name, df in [("events", events), ("category_tree", cats), ("item_properties", props)]:
    print(f"{name}: {df.shape[0]:,} rows x {df.shape[1]} cols")

## 2. Events — the heart of the pipeline

Each row is one visitor action. `transactionid` only exists for purchase events.

In [0]:
events.head()

In [0]:
events["event"].value_counts().plot(kind="bar", title="Event type distribution (sample)")
plt.ylabel("count")
plt.tight_layout()
plt.show()

## 3. Timestamps — milliseconds, not seconds

The sample max is `1433221332117` — **13 digits = Unix milliseconds**. A seconds-based
converter would produce year 47,592 and break the ETL. Divide by 1000 first.

In [0]:
events["ts_sec"] = events["timestamp"] / 1000
events["ts_date"] = pd.to_datetime(events["ts_sec"], unit="s")
print("Range:", events["ts_date"].min(), "->", events["ts_date"].max())

## 4. Missing values

`transactionid` is missing for non-purchase events — expected, but the ETL must handle it.

In [0]:
events.isna().sum()

## 5. `item_properties` — a change log, not a snapshot

The same item appears **many times** as its attributes change over time.

In [0]:
props["property"].value_counts()

In [0]:
# Same item, same property, changing over time:
props[props["itemid"] == props["itemid"].iloc[0]].head(10)

## 6. Hashed values — 90%+ unusable

Values look like `24214 44214 n2017.000` — hashed for privacy. Only `categoryid` and
`available` are readable, which is why the pipeline extracts **only categoryid**.

In [0]:
props["value"].sample(10, random_state=42).tolist()

## 7. `category_tree` — the category hierarchy

`parentid` of 0 (or null) = root category. Let's check the shape and depth.

In [0]:
cats.head()

In [0]:
print("Total categories:", len(cats))
print("Root categories (no parent):", (cats["parentid"].isna()).sum())

## Key takeaways for the pipeline

1. **Timestamps are milliseconds** — `ts/1000` before any date conversion.
2. **`item_properties` is a change log** — dedupe to *latest value per item per property*
   (`max(timestamp)` per item) before joining; categoryid is the only useful property.
3. **`transactionid` is nullable** — only present on `transaction` events.
4. **Event types**: view dominates; transaction is the rarest (typical e-commerce).